In [29]:
import pandas as pd
from PIL import Image
import numpy as np
import os

# Correct way to handle Windows paths
metadata_path = r'Brahmi2\dataset\metadata.csv'

# Alternative formats:
# metadata_path = 'Brahmi2/dataset/metadata.csv'
# metadata_path = 'Brahmi2\\dataset\\metadata.csv'

try:
    # Load metadata
    train_meta = pd.read_csv(metadata_path)
    
    # Create character vocabulary
    all_texts = ' '.join(train_meta['text'].astype(str).values)  # Added astype(str) for safety
    chars = sorted(list(set(all_texts)))
    char_to_num = {c:i for i,c in enumerate(chars)}
    num_to_char = {i:c for i,c in enumerate(chars)}
    
    # Count unique characters
    print(f"Total unique characters: {len(chars)}")
    print("Sample characters:", chars[:10])  # Print first 10 characters
    
    # Verify data loading
    print("\nFirst 5 entries:")
    print(train_meta.head())
    
except FileNotFoundError:
    print(f"Error: File not found at {metadata_path}")
    print("Check if the file exists at this path.")
except Exception as e:
    print(f"An error occurred: {str(e)}")

Total unique characters: 110
Sample characters: [' ', '𑀀', '𑀁', '𑀂', '𑀃', '𑀄', '𑀅', '𑀆', '𑀇', '𑀈']

First 5 entries:
                      file    text  \
0  train/brahmi_000000.png       𑀧   
1  train/brahmi_000001.png       𑀽   
2  train/brahmi_000002.png      𑁜𑀷   
3    val/brahmi_000003.png  𑀩𑀏𑁌𑁃𑀚𑀞   
4  train/brahmi_000004.png  𑀺𑀦𑁗𑁒𑀴𑀻   

                                           unicode  
0                                          U+11027  
1                                          U+1103D  
2                                  U+1105C U+11037  
3  U+11029 U+1100F U+1104C U+11043 U+1101A U+1101E  
4  U+1103A U+11026 U+11057 U+11052 U+11034 U+1103B  


In [35]:
# 1. Verify image paths and sizes
def verify_images(metadata, base_path):
    for idx, row in metadata.iterrows():
        img_path = os.path.join(base_path, row['file'])
        try:
            with Image.open(img_path) as img:
                if idx < 3:  # Print first 3 image sizes
                    print(f"Image: {row['file']}, Size: {img.size}, Mode: {img.mode}")
        except Exception as e:
            print(f"Error loading {img_path}: {str(e)}")

dataset_base = r'Brahmi2\dataset'
verify_images(train_meta, dataset_base)

# 2. Analyze text length distribution
text_lengths = train_meta['text'].str.len()
print("\nText length distribution:")
print(text_lengths.describe())

Image: train/brahmi_000002.png, Size: (256, 64), Mode: L
Image: train/brahmi_000001.png, Size: (256, 64), Mode: L

Text length distribution:
count    8000.000000
mean        2.968250
std         1.912838
min         1.000000
25%         1.000000
50%         3.000000
75%         4.000000
max         7.000000
Name: text, dtype: float64


In [36]:
from torch.utils.data import Dataset
from torchvision import transforms
import torch

class BrahmiDataset(Dataset):
    def __init__(self, metadata, base_path, char_to_num, img_height=64, img_width=256):
        self.metadata = metadata
        self.base_path = base_path
        self.char_to_num = char_to_num
        self.transform = transforms.Compose([
            transforms.ToTensor(),  # Converts to [0,1] range
            transforms.Normalize(mean=[0.5], std=[0.5])  # Normalizes to [-1,1]
        ])
        self.img_height = img_height
        self.img_width = img_width
        
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        img_path = os.path.join(self.base_path, row['file'])
        
        # Load and preprocess image
        image = Image.open(img_path).convert('L')  # Convert to grayscale
        image = image.resize((self.img_width, self.img_height))  # Resize to consistent dimensions
        image = self.transform(image)
        
        # Prepare text target
        text = row['text']
        target = [self.char_to_num[c] for c in text]
        target_length = torch.tensor(len(target), dtype=torch.long)
        target = torch.tensor(target, dtype=torch.long)
        
        return {
            'image': image,
            'target': target,
            'target_length': target_length,
            'text': text  # For debugging
        }

In [37]:
# Initialize dataset
dataset = BrahmiDataset(
    metadata=train_meta[train_meta['file'].str.startswith('train/')],  # Filter train samples
    base_path=dataset_base,  # Your dataset root path
    char_to_num=char_to_num  # From previous step
)

# Test one sample
sample = dataset[0]
print(f"Image tensor shape: {sample['image'].shape}")
print(f"Target sequence: {sample['target']}")
print(f"Target text: {sample['text']}")
print(f"Sequence length: {sample['target_length']}")

Image tensor shape: torch.Size([1, 64, 256])
Target sequence: tensor([14, 64, 51])
Target text: 𑀍𑀿𑀲
Sequence length: 3


In [38]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Split metadata into train and validation (if not already split)
train_meta, val_meta = train_test_split(train_meta, test_size=0.2, random_state=42)

# Initialize datasets
train_dataset = BrahmiDataset(
    metadata=train_meta,
    base_path=dataset_base,
    char_to_num=char_to_num
)

val_dataset = BrahmiDataset(
    metadata=val_meta,
    base_path=dataset_base,
    char_to_num=char_to_num
)

# Create DataLoaders
batch_size = 32  # Adjust based on your GPU memory
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda batch: {
        'images': torch.stack([item['image'] for item in batch]),
        'targets': torch.cat([item['target'] for item in batch]),
        'target_lengths': torch.stack([item['target_length'] for item in batch]),
        'texts': [item['text'] for item in batch]
    }
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda batch: {
        'images': torch.stack([item['image'] for item in batch]),
        'targets': torch.cat([item['target'] for item in batch]),
        'target_lengths': torch.stack([item['target_length'] for item in batch]),
        'texts': [item['text'] for item in batch]
    }
)

# Test one batch
sample_batch = next(iter(train_loader))
print("\nBatch shapes:")
print(f"Images: {sample_batch['images'].shape}")  # Should be [batch_size, 1, 64, 256]
print(f"Targets: {sample_batch['targets'].shape}")  # 1D tensor of all characters in batch
print(f"Target lengths: {sample_batch['target_lengths'].shape}")  # 1D tensor of lengths
print(f"Sample texts: {sample_batch['texts'][:3]}")  # First 3 texts in batch


Batch shapes:
Images: torch.Size([32, 1, 64, 256])
Targets: torch.Size([77])
Target lengths: torch.Size([32])
Sample texts: ['𑀤', '𑀼', '𑁘']


In [39]:
class BrahmiCRNN(nn.Module):
    def __init__(self, num_chars):
        super(BrahmiCRNN, self).__init__()
        
        # CNN Feature Extractor
        self.cnn = nn.Sequential(
            # Input: [B, 1, 64, 256]
            nn.Conv2d(1, 64, 3, stride=1, padding=1), nn.ReLU(), 
            nn.MaxPool2d(2, 2),  # [B, 64, 32, 128]
            
            nn.Conv2d(64, 128, 3, stride=1, padding=1), nn.ReLU(), 
            nn.MaxPool2d(2, 2),  # [B, 128, 16, 64]
            
            nn.Conv2d(128, 256, 3, stride=1, padding=1), nn.BatchNorm2d(256), nn.ReLU(),  # [B, 256, 16, 64]
            
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(),  # [B, 256, 16, 64]
            nn.MaxPool2d((2, 2)),  # [B, 256, 8, 32]
            
            nn.Conv2d(256, 512, 3, stride=1, padding=1), nn.BatchNorm2d(512), nn.ReLU(),  # [B, 512, 8, 32]
            nn.MaxPool2d((2, 1)),  # [B, 512, 4, 32]
            
            nn.Conv2d(512, 512, (4, 1), stride=1), nn.ReLU()  # [B, 512, 1, 32]
        )
        
        # RNN Sequence Model
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.25
        )
        
        # Output Layer
        self.fc = nn.Linear(512, num_chars + 1)  # 512 = 256*2 (bidirectional)

    def forward(self, x):
        # CNN
        x = self.cnn(x)  # [B, 512, 1, 32]
        
        # Remove height dimension
        x = x.squeeze(2)  # [B, 512, 32]
        
        # Prepare for RNN: [seq_len, batch, features]
        x = x.permute(2, 0, 1)  # [32, B, 512]
        
        # RNN
        x, _ = self.rnn(x)  # [32, B, 512]
        
        # Output
        x = self.fc(x)  # [32, B, num_chars + 1]
        return x

In [40]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BrahmiCRNN(num_chars=len(chars)).to(device)

# Test with a sample batch
sample_batch['images'] = sample_batch['images'].to(device)
outputs = model(sample_batch['images'])

print("\nModel output shapes:")
print(f"CNN-RNN outputs: {outputs.shape}")  # Should be [32, batch_size, num_chars+1]
print(f"Sample output at t=0: {outputs[0, 0, :5].detach().cpu().numpy()}")  # First 5 values


Model output shapes:
CNN-RNN outputs: torch.Size([32, 32, 111])
Sample output at t=0: [8.9921756e-03 6.1033294e-05 5.0000317e-02 1.2607342e-02 2.0192768e-02]


In [41]:
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence

# Initialize CTC Loss (blank token is last index)
ctc_loss = nn.CTCLoss(blank=len(chars)).to(device)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Calculate input lengths (fixed at 32 for this architecture)
input_lengths = torch.full(
    size=(sample_batch['images'].size(0),), 
    fill_value=32, 
    dtype=torch.long
).to(device)

In [42]:
def train_batch(model, batch, optimizer, ctc_loss, device):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    images = batch['images'].to(device)
    outputs = model(images)  # [32, B, 111]
    
    # Prepare targets and lengths
    targets = batch['targets'].to(device)  # Flattened targets
    target_lengths = batch['target_lengths'].to(device)
    
    # CTC Loss calculation (FIXED SYNTAX)
    loss = ctc_loss(
        outputs.log_softmax(2),  # [T, N, C] with log probabilities
        targets,
        input_lengths=torch.full((outputs.size(1),), outputs.size(0), dtype=torch.long).to(device),
        target_lengths=target_lengths
    )
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    return loss.item()

In [43]:
import time
from collections import defaultdict

# Initialize
model = BrahmiCRNN(num_chars=len(chars)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
ctc_loss = nn.CTCLoss(blank=len(chars)).to(device)

# Training history storage
history = defaultdict(list)
start_time = time.time()

# Training loop
for epoch in range(1, 11):
    model.train()
    epoch_loss = 0
    batch_times = []
    
    for i, batch in enumerate(train_loader):
        batch_start = time.time()
        
        # Forward pass
        optimizer.zero_grad()
        images = batch['images'].to(device)
        targets = batch['targets'].to(device)
        target_lengths = batch['target_lengths'].to(device)
        
        outputs = model(images)
        
        # Loss calculation
        loss = ctc_loss(
            outputs.log_softmax(2),
            targets,
            input_lengths=torch.full((outputs.size(1),), outputs.size(0), dtype=torch.long).to(device),
            target_lengths=target_lengths
        )
        
        # Backward pass
        loss.backward()
        
        # Gradient monitoring (critical!)
        grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
        max_grad = max(grad_norms) if grad_norms else 0
        
        optimizer.step()
        
        # Record metrics
        batch_time = time.time() - batch_start
        batch_times.append(batch_time)
        epoch_loss += loss.item()
        
        if i % 50 == 0:
            print(f"Batch {i} | Loss: {loss.item():.4f} | Grad: {max_grad:.4f} | {batch_time:.2f}s/batch")
            history['batch_loss'].append(loss.item())
            history['grad_norm'].append(max_grad)
    
    # Epoch statistics
    avg_epoch_loss = epoch_loss / len(train_loader)
    avg_batch_time = sum(batch_times)/len(batch_times)
    history['epoch_loss'].append(avg_epoch_loss)
    history['epoch_time'].append(avg_batch_time)
    
    print(f"\nEpoch {epoch} Summary:")
    print(f"Avg Loss: {avg_epoch_loss:.4f}")
    print(f"Avg Batch Time: {avg_batch_time:.2f}s")
    print(f"Max Gradient: {max(grad_norms):.4f}" if grad_norms else "No Gradients")
    print("-" * 50)
    
    # Save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': avg_epoch_loss,
    }, f"epoch_{epoch}_checkpoint.pth")

# Final report
total_time = time.time() - start_time
print(f"\nTraining Complete in {total_time/60:.1f} minutes")
print(f"Final Loss: {history['epoch_loss'][-1]:.4f}")
print(f"Loss Reduction: {history['epoch_loss'][0]/history['epoch_loss'][-1]:.1f}x improvement")

Batch 0 | Loss: 82.0440 | Grad: 15.5736 | 1.56s/batch
Batch 50 | Loss: 5.5723 | Grad: 3.2948 | 1.10s/batch
Batch 100 | Loss: 5.6745 | Grad: 3.5828 | 1.06s/batch
Batch 150 | Loss: 5.6257 | Grad: 1.7832 | 1.14s/batch

Epoch 1 Summary:
Avg Loss: 6.4980
Avg Batch Time: 1.20s
Max Gradient: 1.7168
--------------------------------------------------
Batch 0 | Loss: 5.6634 | Grad: 2.1398 | 1.46s/batch
Batch 50 | Loss: 5.4427 | Grad: 2.2253 | 1.46s/batch
Batch 100 | Loss: 5.6456 | Grad: 2.8382 | 1.10s/batch
Batch 150 | Loss: 5.6309 | Grad: 2.5876 | 1.31s/batch

Epoch 2 Summary:
Avg Loss: 5.6007
Avg Batch Time: 1.32s
Max Gradient: 2.7920
--------------------------------------------------
Batch 0 | Loss: 5.4651 | Grad: 2.1442 | 1.27s/batch
Batch 50 | Loss: 5.6769 | Grad: 2.0376 | 1.28s/batch
Batch 100 | Loss: 5.8002 | Grad: 3.5441 | 1.29s/batch
Batch 150 | Loss: 5.5485 | Grad: 1.6907 | 1.34s/batch

Epoch 3 Summary:
Avg Loss: 5.5905
Avg Batch Time: 1.31s
Max Gradient: 1.3857
-----------------------

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# 1. Prepare test data (if you don't have dedicated test set)
# Option A: Use your validation set as test
test_loader = val_loader  # You already have val_loader defined

# Option B: Create new test split from training data
# First load your original metadata
metadata = pd.read_csv(r'Brahmi2\dataset\metadata.csv')  # Your path

# Split into train+val and test (80-10-10)
trainval_meta, test_meta = train_test_split(metadata, test_size=0.1, random_state=42)
train_meta, val_meta = train_test_split(trainval_meta, test_size=0.1, random_state=42)  # 0.1 of 0.9 = 9% val

# Create test dataset
test_dataset = BrahmiDataset(
    metadata=test_meta,
    base_path=dataset_base,  # Your dataset root path
    char_to_num=char_to_num  # From your training setup
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda batch: {
        'images': torch.stack([item['image'] for item in batch]),
        'targets': torch.cat([item['target'] for item in batch]),
        'target_lengths': torch.stack([item['target_length'] for item in batch]),
        'texts': [item['text'] for item in batch]
    }
)

# 2. Evaluation function with CER (Character Error Rate)
def decode_predictions(outputs, num_to_char):
    """Greedy decoder for CTC outputs"""
    _, max_indices = torch.max(outputs, 2)
    texts = []
    for i in range(max_indices.size(1)):
        indices = max_indices[:, i]
        chars = [num_to_char[idx.item()] for idx in indices if idx != len(num_to_char)]
        texts.append(''.join(chars))
    return texts

def calculate_cer(pred, true):
    """Compute Character Error Rate"""
    import numpy as np
    # Create distance matrix
    d = np.zeros((len(true)+1, len(pred)+1))
    d[0,:] = np.arange(len(pred)+1)
    d[:,0] = np.arange(len(true)+1)
    
    for i in range(1, len(true)+1):
        for j in range(1, len(pred)+1):
            cost = 0 if true[i-1] == pred[j-1] else 1
            d[i,j] = min(d[i-1,j]+1, d[i,j-1]+1, d[i-1,j-1]+cost)
    
    return d[-1,-1] / len(true)

def evaluate(model, loader, ctc_loss, num_to_char, device):
    model.eval()
    total_loss, total_cer, correct, total = 0, 0, 0, 0
    error_samples = []
    
    with torch.no_grad():
        for batch in loader:
            # Forward pass
            outputs = model(batch['images'].to(device))
            
            # Loss calculation
            loss = ctc_loss(
                outputs.log_softmax(2),
                batch['targets'].to(device),
                input_lengths=torch.full((outputs.size(1),), outputs.size(0), dtype=torch.long).to(device),
                target_lengths=batch['target_lengths'].to(device)
            )
            total_loss += loss.item()
            
            # Decoding and metrics
            pred_texts = decode_predictions(outputs, num_to_char)
            true_texts = batch['texts']
            
            for pred, true in zip(pred_texts, true_texts):
                cer = calculate_cer(pred, true)
                total_cer += cer
                total += 1
                correct += (pred == true)
                if pred != true and len(error_samples) < 5:
                    error_samples.append((true, pred, cer))
    
    return {
        'avg_loss': total_loss / len(loader),
        'accuracy': correct / total,
        'avg_cer': total_cer / total,
        'error_samples': error_samples
    }

# 3. Run evaluation
eval_results = evaluate(
    model=model,
    loader=test_loader,
    ctc_loss=ctc_loss,
    num_to_char=num_to_char,
    device=device
)

print(f"""
Evaluation Results:
- Avg Loss: {eval_results['avg_loss']:.4f}
- Accuracy: {eval_results['accuracy']*100:.2f}%
- Character Error Rate: {eval_results['avg_cer']*100:.2f}%
""")

print("\nTop Error Samples:")
for i, (true, pred, cer) in enumerate(eval_results['error_samples']):
    print(f"{i+1}. True: '{true}' | Pred: '{pred}' | CER: {cer*100:.1f}%")


Evaluation Results:
- Avg Loss: 0.3893
- Accuracy: 76.20%
- Character Error Rate: 8.55%


Top Error Samples:
1. True: '𑀽𑀰' | Pred: '𑀽' | CER: 50.0%
2. True: '𑁙𑀜𑁁' | Pred: '𑁙𑀜𑁀' | CER: 33.3%
3. True: '𑀤𑀼' | Pred: '𑀤' | CER: 50.0%
4. True: '𑁫𑀖𑀶𑀤' | Pred: '𑁫𑀖𑀶𑁖' | CER: 25.0%
5. True: '𑀟𑁀𑀑𑀴𑁁𑀉' | Pred: '𑀟𑀑𑀴𑀉' | CER: 33.3%


In [1]:
import torch
import torch.nn as nn

# --------------------------
# Define your CRNN model
# --------------------------
class BrahmiCRNN(nn.Module):
    def __init__(self, num_chars):
        super(BrahmiCRNN, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # [B, 64, 32, 128]

            nn.Conv2d(64, 128, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # [B, 128, 16, 64]

            nn.Conv2d(128, 256, 3, stride=1, padding=1), nn.BatchNorm2d(256), nn.ReLU(),

            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d((2, 2)),  # [B, 256, 8, 32]

            nn.Conv2d(256, 512, 3, stride=1, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1)),  # [B, 512, 4, 32]

            nn.Conv2d(512, 512, (4, 1), stride=1), nn.ReLU()  # [B, 512, 1, 32]
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.25
        )

        self.fc = nn.Linear(512, num_chars + 1)

    def forward(self, x):
        x = self.cnn(x)              # [B, 512, 1, 32]
        x = x.squeeze(2)             # [B, 512, 32]
        x = x.permute(2, 0, 1)       # [32, B, 512]
        x, _ = self.rnn(x)           # [32, B, 512]
        x = self.fc(x)               # [32, B, num_chars+1]
        return x


# --------------------------
# ONNX Export Script
# --------------------------
# Load model
num_chars = 110  # or len(chars) from your training
model = BrahmiCRNN(num_chars=num_chars)
checkpoint = torch.load("epoch_10_checkpoint.pth", map_location='cpu')
model.load_state_dict(checkpoint['model_state'])
model.eval()

# Create dummy input matching training image size: [B, C, H, W] = [1, 1, 64, 256]
dummy_input = torch.randn(1, 1, 64, 256)

# Export to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "brahmi_ocr_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {1: "batch_size"}},
    opset_version=11
)

print("✅ Export complete! Saved as brahmi_ocr_model.onnx")


myenv\lib\site-packages\torch\onnx\symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


✅ Export complete! Saved as brahmi_ocr_model.onnx


In [3]:
import onnxruntime as ort
import numpy as np
import cv2
import torch

# Load ONNX model
onnx_model_path = "brahmi_ocr_model.onnx"
session = ort.InferenceSession(onnx_model_path)

# Input/output names
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Character list (make sure it's the same used in training)
chars = [...]  # <- REPLACE with your actual character list

def preprocess_image(img_path):
    # Load image (grayscale)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (256, 64))  # Resize to (W, H) = (256, 64)
    img = img.astype(np.float32) / 255.0  # Normalize to [0, 1]
    img = np.expand_dims(img, axis=0)  # [1, 64, 256]
    img = np.expand_dims(img, axis=0)  # [1, 1, 64, 256]
    return img

def greedy_decoder(output_tensor):
    pred_indices = np.argmax(output_tensor, axis=2)  # [T, B]
    pred_indices = pred_indices[:, 0]  # Take batch 0
    decoded = []
    prev = -1
    for idx in pred_indices:
        if idx != prev and idx != len(chars):  # Ignore repeated and blank
            decoded.append(chars[idx])
        prev = idx
    return ''.join(decoded)

# Load and preprocess your image
image_path = r"OCR\4_binary.png"
input_image = preprocess_image(image_path)

# Run inference
outputs = session.run([output_name], {input_name: input_image})[0]  # shape: [T, B, num_classes]
print("Raw ONNX output shape:", outputs.shape)

def greedy_decoder(output):
    pred_indices = np.argmax(output, axis=2).squeeze(1)
    decoded = []
    prev = -1
    for idx in pred_indices:
        if idx != prev and idx != len(chars):  # CTC blank
            if idx < len(chars):              # Valid index
                decoded.append(chars[idx])
        prev = idx
    return ''.join(decoded)




Raw ONNX output shape: (32, 1, 111)


In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class BrahmiCRNN(nn.Module):
    def __init__(self, num_chars):
        super(BrahmiCRNN, self).__init__()
        # CNN Layers
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, stride=1, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, stride=1, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, stride=1, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(256, 512, 3, stride=1, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(512, 512, 2, stride=1), nn.ReLU()
        )
        
        # RNN Layers
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.25
        )
        
        # Output Layer
        self.fc = nn.Linear(512, num_chars + 1)  # +1 for CTC blank

    def forward(self, x):
        # CNN
        x = self.cnn(x)
        x = x.squeeze(2)
        x = x.permute(2, 0, 1)
        
        # RNN
        x, _ = self.rnn(x)
        
        # Output
        x = self.fc(x)
        return x

In [ ]:
# 1. Load checkpoint with proper key access
checkpoint = torch.load(checkpoint_path, map_location=device)

# 2. Check available keys
print("Available checkpoint keys:")
print(checkpoint.keys())

# 3. Load using the correct key ('model_state' in your case)
if 'model_state' in checkpoint:
    model.load_state_dict(checkpoint['model_state'])
    print("Successfully loaded from 'model_state' key!")
elif 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print("Successfully loaded from 'model_state_dict' key!")
else:
    raise ValueError("Checkpoint doesn't contain model weights in expected keys")

# 4. Verification
print("\nModel loaded successfully!")
print(f"Model structure:\n{model}")

# Test forward pass
test_input = torch.randn(1, 1, 64, 256).to(device)
with torch.no_grad():
    output = model(test_input)
print(f"\nTest output shape: {output.shape}")

Available checkpoint keys:
dict_keys(['epoch', 'model_state', 'optimizer_state', 'loss'])


RuntimeError: Error(s) in loading state_dict for BrahmiCRNN:
	size mismatch for cnn.16.weight: copying a param with shape torch.Size([512, 512, 4, 1]) from checkpoint, the shape in current model is torch.Size([512, 512, 2, 2]).

In [ ]:
# Verify the loaded model works
test_input = torch.randn(1, 1, 64, 256).to(device)  # Fake image
output = model(test_input)
print(f"Test output shape: {output.shape}")  # Should be [seq_len, 1, num_chars+1]

Test output shape: torch.Size([32, 1, 111])
